In [4]:

import sqlite3
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Use the same database path as your project
DB_PATH = "/Users/Maazuddin/Downloads/inventory.db"

# 2. Load invoice-level records
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        """
        SELECT PONumber, VendorNumber, Quantity, Dollars, Freight
        FROM vendor_invoice
        """,
        conn
    )

print("Original rows:", len(df))

# 3. Check whether PONumber is unique
print("Duplicate PONumber rows:", df["PONumber"].duplicated().sum())

# Do not drop duplicates automatically.
# Investigate them if they exist.

# 4. Convert model columns to numeric
features = ["Dollars", "Quantity"]
target = "Freight"

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Remove rows with missing/non-numeric model values
df = df.dropna(subset=features + [target]).copy()

print("Rows after removing missing values:", len(df))
print("Negative quantity rows:", (df["Quantity"] < 0).sum())
print("Negative dollar rows:", (df["Dollars"] < 0).sum())
print("Negative freight rows:", (df["Freight"] < 0).sum())

# Negative values are reported, not silently deleted.
# Investigate them before deciding whether they are valid.

# 6. Create X and y at invoice level
X = df[features]
y = df[target]

# 7. Hold out 20% of records for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 8. Train the baseline model
model = LinearRegression()
model.fit(X_train, y_train)

# 9. Predict held-out invoices
y_pred = model.predict(X_test)

# 10. Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\n--- Freight Baseline Evaluation ---")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

print("\nModel coefficients:")
print("Intercept:", model.intercept_)
print("Coefficients:", dict(zip(features, model.coef_)))

# 11. Compare with a simple mean-prediction baseline
mean_prediction = [y_train.mean()] * len(y_test)

baseline_mae = mean_absolute_error(y_test, mean_prediction)
baseline_rmse = mean_squared_error(
    y_test, mean_prediction
) ** 0.5
baseline_r2 = r2_score(y_test, mean_prediction)

print("\n--- Mean-Prediction Baseline ---")
print(f"MAE:  {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²:   {baseline_r2:.4f}")



Original rows: 5543
Duplicate PONumber rows: 0
Rows after removing missing values: 5543
Negative quantity rows: 0
Negative dollar rows: 0
Negative freight rows: 0

--- Freight Baseline Evaluation ---
Training rows: 4434
Test rows: 1109
MAE:  24.4592
RMSE: 124.4288
R²:   0.9700

Model coefficients:
Intercept: 5.604873227096903
Coefficients: {'Dollars': np.float64(0.005214217265819658), 'Quantity': np.float64(-0.0019980023189016013)}

--- Mean-Prediction Baseline ---
MAE:  388.0030
RMSE: 718.6298
R²:   -0.0000


In [5]:
# Compare actual and predicted freight
results = pd.DataFrame({
    "Actual_Freight": y_test,
    "Predicted_Freight": y_pred
})

results["Error"] = (
    results["Actual_Freight"] - results["Predicted_Freight"]
)
results["Absolute_Error"] = results["Error"].abs()

print("\nLargest prediction errors:")
print(
    results.sort_values(
        "Absolute_Error", ascending=False
    ).head(10).round(2).to_string(index=False)
)

print("\nFreight target distribution:")
print(df["Freight"].describe().round(2))

print("\nFeature correlations:")
print(
    df[["Dollars", "Quantity", "Freight"]]
    .corr()
    .round(3)
)

print("\nNegative raw predictions:", (y_pred < 0).sum())



Largest prediction errors:
 Actual_Freight  Predicted_Freight   Error  Absolute_Error
        4613.45             938.93 3674.52         3674.52
        1206.72             294.11  912.61          912.61
        4581.05            5093.92 -512.87          512.87
         571.07             111.07  460.00          460.00
        4052.76            4505.79 -453.03          453.03
        4531.33            4112.02  419.31          419.31
        3473.39            3858.14 -384.75          384.75
         408.72              75.67  333.05          333.05
        5855.78            5527.25  328.53          328.53
        3649.71            3368.92  280.79          280.79

Freight target distribution:
count    5543.00
mean      295.95
std       713.59
min         0.02
25%         5.02
50%        24.73
75%       229.66
max      8468.22
Name: Freight, dtype: float64

Feature correlations:
          Dollars  Quantity  Freight
Dollars     1.000     0.964    0.985
Quantity    0.964     1.000   

In [6]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# A. Examine freight as a proportion of invoice dollars
df["Freight_Rate"] = df["Freight"] / df["Dollars"].where(
    df["Dollars"] != 0
)

print("\nFreight as a proportion of invoice dollars:")
print(df["Freight_Rate"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# B. Evaluate a Dollars-only model on the SAME test rows
dollars_model = LinearRegression()
dollars_model.fit(X_train[["Dollars"]], y_train)

dollars_pred = dollars_model.predict(X_test[["Dollars"]])

print("\n--- Dollars-only model ---")
print(f"MAE:  {mean_absolute_error(y_test, dollars_pred):.4f}")
print(f"RMSE: {mean_squared_error(y_test, dollars_pred) ** 0.5:.4f}")
print(f"R²:   {r2_score(y_test, dollars_pred):.4f}")

# C. Compare errors by actual freight size
comparison = pd.DataFrame({
    "Actual_Freight": y_test,
    "Two_Feature_Prediction": y_pred,
    "Dollars_Only_Prediction": dollars_pred
})

comparison["Absolute_Error"] = (
    comparison["Actual_Freight"]
    - comparison["Two_Feature_Prediction"]
).abs()

comparison["Freight_Group"] = pd.cut(
    comparison["Actual_Freight"],
    bins=[-float("inf"), 25, 250, 1000, float("inf")],
    labels=["Up to 25", "25–250", "250–1000", "Above 1000"]
)

print("\nError by actual freight group:")
print(
    comparison.groupby("Freight_Group", observed=False)
    .agg(
        Invoice_Count=("Actual_Freight", "size"),
        Mean_Actual_Freight=("Actual_Freight", "mean"),
        MAE=("Absolute_Error", "mean")
    )
    .round(2)
)



Freight as a proportion of invoice dollars:
count    5543.000000
mean        0.005505
std         0.004801
min         0.004416
50%         0.005000
90%         0.005496
95%         0.005500
99%         0.027211
max         0.115225
Name: Freight_Rate, dtype: float64

--- Dollars-only model ---
MAE:  24.1060
RMSE: 124.7210
R²:   0.9699

Error by actual freight group:
               Invoice_Count  Mean_Actual_Freight     MAE
Freight_Group                                            
Up to 25                 567                 6.79    5.64
25–250                   277               110.54    9.57
250–1000                 176               502.32   35.74
Above 1000                89              2284.69  168.37


In [7]:

# Inspect invoices with unusually high freight rates
inspection = df.copy()

inspection["Freight_Rate"] = (
    inspection["Freight"]
    / inspection["Dollars"].where(inspection["Dollars"] != 0)
)

print("\nTop 15 invoices by freight-to-dollar ratio:")

print(
    inspection.nlargest(15, "Freight_Rate")[
        [
            "PONumber",
            "VendorNumber",
            "Quantity",
            "Dollars",
            "Freight",
            "Freight_Rate"
        ]
    ].to_string(index=False)
)

# Compare the freight rate by vendor
vendor_summary = (
    inspection.groupby("VendorNumber")
    .agg(
        Invoice_Count=("Freight", "size"),
        Median_Freight_Rate=("Freight_Rate", "median"),
        Mean_Freight_Rate=("Freight_Rate", "mean")
    )
    .query("Invoice_Count >= 5")
    .sort_values("Median_Freight_Rate", ascending=False)
)

print("\nVendors with highest median freight rates:")
print(vendor_summary.head(15).round(5))



Top 15 invoices by freight-to-dollar ratio:
 PONumber  VendorNumber  Quantity  Dollars  Freight  Freight_Rate
     8143          4848        12   120.72    13.91      0.115225
     8206         90058        23   186.93    21.04      0.112556
     8172          1273         5   146.80    15.53      0.105790
     8196          4550         7    98.49     9.65      0.097979
     8198          7255        11    66.33     6.09      0.091814
     8121         28776         6   133.26    10.59      0.079469
     8197          7154        11   176.11    12.08      0.068593
     8120          8920        12    55.08     3.62      0.065723
     8137          4466        15   140.55     8.57      0.060975
     8160          4425        24   193.80    11.38      0.058720
     8188          1601        28   393.52    22.66      0.057583
     8201           653       541  5544.89   303.70      0.054771
     8205         90010        24   176.04     9.41      0.053454
     8111         17032        

In [8]:

# Match predictions to the held-out invoice records
test_results = df.loc[X_test.index, [
    "PONumber",
    "VendorNumber",
    "Quantity",
    "Dollars",
    "Freight"
]].copy()

test_results["Predicted_Freight"] = y_pred
test_results["Absolute_Error"] = (
    test_results["Freight"]
    - test_results["Predicted_Freight"]
).abs()

test_results["Freight_Rate"] = (
    test_results["Freight"]
    / test_results["Dollars"].where(
        test_results["Dollars"] != 0
    )
)

# Show high-rate invoices that are actually in the test set
print("\nHighest freight-rate invoices in the test set:")
print(
    test_results.nlargest(15, "Freight_Rate")[
        [
            "PONumber",
            "VendorNumber",
            "Dollars",
            "Quantity",
            "Freight",
            "Predicted_Freight",
            "Absolute_Error",
            "Freight_Rate"
        ]
    ].round(3).to_string(index=False)
)

# Show the invoices with the largest prediction errors
print("\nLargest prediction errors in the test set:")
print(
    test_results.nlargest(15, "Absolute_Error")[
        [
            "PONumber",
            "VendorNumber",
            "Dollars",
            "Quantity",
            "Freight",
            "Predicted_Freight",
            "Absolute_Error",
            "Freight_Rate"
        ]
    ].round(3).to_string(index=False)
)




Highest freight-rate invoices in the test set:
 PONumber  VendorNumber   Dollars  Quantity  Freight  Predicted_Freight  Absolute_Error  Freight_Rate
     8206         90058    186.93        23    21.04              6.534          14.506         0.113
     8172          1273    146.80         5    15.53              6.360           9.170         0.106
     8196          4550     98.49         7     9.65              6.104           3.546         0.098
     8201           653   5544.89       541   303.70             33.436         270.264         0.055
     8205         90010    176.04        24     9.41              6.475           2.935         0.053
     8130          5992    892.90        90    42.40             10.081          32.319         0.047
     8175          2561   5645.24       136   218.18             34.769         183.411         0.039
     8182          8150    111.05         5     3.39              6.174           2.784         0.031
     8109          1590  14298.09 